# CNN Benchmark on Your Own Dataset (PyTorch + Colab) — with Data Augmentation

Trains and compares 17 torchvision CNNs and records **accuracy, precision, recall, F1, training time, inference time, model size, #params, learning rate, epochs, batch size, loss function**.

**Before running**
1. `Runtime → Change runtime type → GPU` (T4 is fine).
2. Put your images in Google Drive in one of these layouts (standard `ImageFolder` format):
   - **Pre-split:** `my_dataset/train/<class>/*.jpg`, `my_dataset/val/<class>/*.jpg`, (optional) `my_dataset/test/<class>/*.jpg`
   - **Single folder:** `my_dataset/<class>/*.jpg` (the notebook makes a stratified 70/15/15 split for you)
3. Edit `DATA_DIR` in the config cell, then **Run all**.

**Augmentation (training set only):** RandomResizedCrop, RandomHorizontalFlip, RandomRotation, ColorJitter, RandomPerspective, GaussianBlur, Normalize, RandomErasing. Every parameter lives in `AUG` in the config cell; set a probability to `0` to disable a transform. Validation/test images only get resize + center-crop + normalize. A preview cell shows what the augmented images look like.

Tip: do a dry run first with `EPOCHS = 1` to make sure all 17 models run on your GPU. Results are saved to Drive after every model, and finished models are skipped if you re-run (so a Colab disconnect doesn't lose everything).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
nvidia-smi -L

In [ ]:
import os, gc, time, json, copy, random, shutil, traceback, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

# ======================= CONFIG (edit me) =======================
DATA_DIR  = '/content/drive/MyDrive/Thesis/prelim'            # your dataset folder
OUT_DIR   = '/content/drive/MyDrive/Thesis/prelim/results'  # results/checkpoints are saved here
COPY_DATA_TO_LOCAL = True     # copy dataset to /content (much faster than reading from Drive)

EPOCHS       = 20
BATCH_SIZE   = 32             # lower to 16 if you hit CUDA OOM on big models / Inception-v3
LR           = 1e-4           # 1e-4 is good for fine-tuning; use ~1e-3 if PRETRAINED = False
LR_OVERRIDES = {}             # per-model LR, e.g. {'AlexNet': 1e-5, 'VGG-11': 1e-5}
WEIGHT_DECAY = 1e-4
PRETRAINED   = True           # True = ImageNet weights (transfer learning); False = from scratch
USE_AMP      = True           # mixed precision (faster, less memory on GPU)
NUM_WORKERS  = 2
SPLIT        = (0.70, 0.15, 0.15)   # only used when DATA_DIR has no train/val folders
PATIENCE     = None           # early stopping on val accuracy, e.g. 5 (None = off)
MEASURE_CPU_LATENCY = True
SEED         = 42
LOSS_NAME    = 'CrossEntropyLoss'
OPTIM_NAME   = 'AdamW'

# ---- data augmentation (applied to TRAIN images only) ----
AUG = dict(
    crop_scale=(0.5, 1.0),        # RandomResizedCrop: fraction of the image area kept
    crop_ratio=(3/4, 4/3),        # RandomResizedCrop: aspect ratio range
    hflip_p=0.5,                  # RandomHorizontalFlip
    rotation_deg=20,              # RandomRotation: +/- degrees
    jitter=dict(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),  # ColorJitter
    jitter_p=0.8,                 # chance ColorJitter is applied
    perspective_distortion=0.3,   # RandomPerspective strength
    perspective_p=0.3,
    blur_sigma=(0.1, 2.0),        # GaussianBlur sigma range
    blur_p=0.3,
    erase_p=0.25,                 # RandomErasing (applied after Normalize)
    erase_scale=(0.02, 0.2),      # fraction of the image area erased
    erase_ratio=(0.3, 3.3),
)
# ================================================================

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

for sub in ('checkpoints', 'history', 'reports'):
    os.makedirs(f'{OUT_DIR}/{sub}', exist_ok=True)

# the 17 models: label -> torchvision name, input size, extra kwargs
MODELS = {
    'AlexNet':            dict(fn='alexnet'),
    'VGG-11':             dict(fn='vgg11'),
    'GoogLeNet':          dict(fn='googlenet', has_aux=True),
    'ResNet-18':          dict(fn='resnet18'),
    'Inception-v3':       dict(fn='inception_v3', size=299, has_aux=True),
    'SqueezeNet 1.1':     dict(fn='squeezenet1_1'),
    'Wide ResNet-50-2':   dict(fn='wide_resnet50_2'),
    'DenseNet-121':       dict(fn='densenet121'),
    'ResNeXt-50 32x4d':   dict(fn='resnext50_32x4d'),
    'MobileNetV2':        dict(fn='mobilenet_v2'),
    'ShuffleNetV2 x0.5':  dict(fn='shufflenet_v2_x0_5'),
    'MNASNet 0.5':        dict(fn='mnasnet0_5'),
    'EfficientNet-B0':    dict(fn='efficientnet_b0'),
    'MobileNetV3-Small':  dict(fn='mobilenet_v3_small'),
    'RegNetY-400MF':      dict(fn='regnet_y_400mf'),
    'EfficientNetV2-S':   dict(fn='efficientnet_v2_s'),
    'ConvNeXt-Tiny':      dict(fn='convnext_tiny'),
}

## Data

In [ ]:
# ---- optionally copy dataset from Drive to local disk for speed ----
if COPY_DATA_TO_LOCAL:
    LOCAL_DIR = '/content/dataset_local'
    if not os.path.exists(LOCAL_DIR):
        print('Copying dataset to local disk...')
        shutil.copytree(DATA_DIR, LOCAL_DIR)
    DATA_ROOT = LOCAL_DIR
else:
    DATA_ROOT = DATA_DIR

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def make_transforms(size):
    k = 2 * int(size * 0.02) + 1      # odd blur kernel that scales with image size (9 @224, 11 @299)
    train_tf = transforms.Compose([
        # ---- PIL-image transforms ----
        transforms.RandomResizedCrop(size, scale=AUG['crop_scale'], ratio=AUG['crop_ratio']),
        transforms.RandomHorizontalFlip(p=AUG['hflip_p']),
        transforms.RandomRotation(AUG['rotation_deg']),
        transforms.RandomApply([transforms.ColorJitter(**AUG['jitter'])], p=AUG['jitter_p']),
        transforms.RandomPerspective(distortion_scale=AUG['perspective_distortion'], p=AUG['perspective_p']),
        transforms.RandomApply([transforms.GaussianBlur(kernel_size=k, sigma=AUG['blur_sigma'])],
                               p=AUG['blur_p']),
        # ---- tensor transforms ----
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
        transforms.RandomErasing(p=AUG['erase_p'], scale=AUG['erase_scale'],
                                 ratio=AUG['erase_ratio'], value='random'),
    ])
    # validation / test: NO random augmentation
    eval_tf = transforms.Compose([
        transforms.Resize(int(size * 1.14)),
        transforms.CenterCrop(size),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD),
    ])
    return train_tf, eval_tf

HAS_SPLIT_DIRS = all(os.path.isdir(os.path.join(DATA_ROOT, s)) for s in ('train', 'val'))
_SPLIT_IDX = {}   # computed once so every model sees the exact same split

def _get_split_idx():
    if not _SPLIT_IDX:
        targets = np.array(datasets.ImageFolder(DATA_ROOT).targets)
        idx = np.arange(len(targets))
        tr, tmp = train_test_split(idx, test_size=1 - SPLIT[0], stratify=targets, random_state=SEED)
        val_share = SPLIT[1] / (SPLIT[1] + SPLIT[2])
        va, te = train_test_split(tmp, test_size=1 - val_share, stratify=targets[tmp], random_state=SEED)
        _SPLIT_IDX.update(train=tr, val=va, test=te)
    return _SPLIT_IDX

from PIL import Image, UnidentifiedImageError

def _safe_loader(path):
    """Like torchvision's default_loader, but returns a black placeholder
    instead of crashing the whole run on a corrupt/unreadable image file."""
    try:
        with open(path, 'rb') as f:
            img = Image.open(f)
            return img.convert('RGB')
    except (UnidentifiedImageError, OSError) as e:
        if path not in _BAD_FILES:
            _BAD_FILES.add(path)
            print(f'  [WARN] unreadable image, using black placeholder instead: {path} ({e})')
        return Image.new('RGB', (256, 256), color=(0, 0, 0))

_BAD_FILES = set()

def make_loaders(size):
    train_tf, eval_tf = make_transforms(size)
    if HAS_SPLIT_DIRS:
        tr = datasets.ImageFolder(f'{DATA_ROOT}/train', train_tf, loader=_safe_loader)
        va = datasets.ImageFolder(f'{DATA_ROOT}/val', eval_tf, loader=_safe_loader)
        test_dir = 'test' if os.path.isdir(f'{DATA_ROOT}/test') else 'val'
        te = datasets.ImageFolder(f'{DATA_ROOT}/{test_dir}', eval_tf, loader=_safe_loader)
        classes = tr.classes
    else:
        s = _get_split_idx()
        full_train = datasets.ImageFolder(DATA_ROOT, train_tf, loader=_safe_loader)
        full_eval = datasets.ImageFolder(DATA_ROOT, eval_tf, loader=_safe_loader)
        tr, va, te = Subset(full_train, s['train']), Subset(full_eval, s['val']), Subset(full_eval, s['test'])
        classes = full_train.classes

    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True,
              persistent_workers=NUM_WORKERS > 0)
    return (DataLoader(tr, shuffle=True, drop_last=len(tr) > BATCH_SIZE, **kw),
            DataLoader(va, shuffle=False, **kw),
            DataLoader(te, shuffle=False, **kw),
            classes)

_tl, _vl, _tel, CLASSES = make_loaders(224)
print(f'Classes ({len(CLASSES)}): {CLASSES}')
print(f'Train: {len(_tl.dataset)} | Val: {len(_vl.dataset)} | Test: {len(_tel.dataset)}')
if HAS_SPLIT_DIRS and not os.path.isdir(f'{DATA_ROOT}/test'):
    print('WARNING: no test/ folder found -> val/ is being used as the test set.')
del _tl, _vl, _tel

### Preview the augmentations
Shows one training image passed through the augmentation pipeline 8 times (re-run to see different samples).

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

train_tf, _ = make_transforms(224)
if HAS_SPLIT_DIRS:
    base = datasets.ImageFolder(f'{DATA_ROOT}/train')
else:
    base = datasets.ImageFolder(DATA_ROOT)
img_path, _ = base.samples[random.randrange(len(base.samples))]
img = Image.open(img_path).convert('RGB')

mean, std = torch.tensor(MEAN).view(3, 1, 1), torch.tensor(STD).view(3, 1, 1)
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes[0, 0].imshow(img); axes[0, 0].set_title('original')
for a in axes.flat[1:]:
    t = train_tf(img) * std + mean          # undo Normalize so the image displays correctly
    a.imshow(t.clamp(0, 1).permute(1, 2, 0).numpy())
for a in axes.flat: a.axis('off')
plt.tight_layout(); plt.show()

## Model builder, training and evaluation helpers

In [ ]:
def build_model(cfg, num_classes):
    weights = 'DEFAULT' if PRETRAINED else None
    kwargs = dict(cfg.get('kwargs', {}))
    if cfg.get('has_aux'):
        # torchvision requires aux_logits=True to load the pretrained checkpoint
        # (it contains the aux-classifier weights); we disable/remove the aux
        # heads ourselves right after construction instead.
        kwargs['aux_logits'] = True
        if not PRETRAINED:
            kwargs['aux_logits'] = False  # fine to skip aux heads when training from scratch
    model = models.get_model(cfg['fn'], weights=weights, **kwargs)
    if cfg.get('has_aux') and PRETRAINED:
        # drop the aux classifiers so forward() returns plain logits, like aux_logits=False
        if hasattr(model, 'aux1'):
            model.aux1 = None
        if hasattr(model, 'aux2'):
            model.aux2 = None
        if hasattr(model, 'AuxLogits'):
            model.AuxLogits = None
        model.aux_logits = False
    return replace_head(model, num_classes)

def replace_head(model, num_classes):
    # SqueezeNet's classifier is a 1x1 conv, not a Linear layer
    if isinstance(model, models.SqueezeNet):
        model.classifier[1] = nn.Conv2d(model.classifier[1].in_channels, num_classes, kernel_size=1)
        model.num_classes = num_classes
        return model
    # every other model: swap its last nn.Linear
    last = None
    for name, m in model.named_modules():
        if isinstance(m, nn.Linear):
            last = name
    parent_name, _, child = last.rpartition('.')
    parent = model.get_submodule(parent_name) if parent_name else model
    old = model.get_submodule(last)
    new = nn.Linear(old.in_features, num_classes)
    if isinstance(parent, (nn.Sequential, nn.ModuleList)):
        parent[int(child)] = new
    else:
        setattr(parent, child, new)
    return model

def get_logits(out):
    # Inception/GoogLeNet can return a namedtuple in train mode
    if hasattr(out, 'logits'):
        return out.logits
    if isinstance(out, tuple):
        return out[0]
    return out

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def sync():
    if DEVICE.type == 'cuda':
        torch.cuda.synchronize()

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    loss_sum, n = 0.0, 0
    preds, tgts = [], []
    amp_on = USE_AMP and DEVICE.type == 'cuda'
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_on):
                logits = get_logits(model(x))
                loss = criterion(logits, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            loss_sum += loss.item() * x.size(0)
            n += x.size(0)
            preds.append(logits.argmax(1).cpu())
            tgts.append(y.cpu())
    preds, tgts = torch.cat(preds).numpy(), torch.cat(tgts).numpy()
    return loss_sum / n, (preds == tgts).mean(), preds, tgts

@torch.no_grad()
def measure_latency(model, size, device, warmup=20, runs=100):
    # single-image (batch=1) FP32 latency in milliseconds
    model = model.to(device).eval()
    x = torch.randn(1, 3, size, size, device=device)
    for _ in range(warmup):
        model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(runs):
        model(x)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - t0) / runs * 1000

In [ ]:
def train_model(label, cfg):
    size = cfg.get('size', 224)
    lr = LR_OVERRIDES.get(label, LR)
    set_seed(SEED)

    train_loader, val_loader, test_loader, classes = make_loaders(size)
    model = build_model(cfg, len(classes)).to(DEVICE)
    n_params, n_trainable = count_params(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and DEVICE.type == 'cuda')

    ckpt_path = f'{OUT_DIR}/checkpoints/{label.replace(" ", "_").replace("/", "-")}.pth'
    best_acc, best_epoch, bad_epochs = -1.0, 0, 0
    train_time, history = 0.0, []

    print(f'\n===== {label} | params {n_params/1e6:.2f}M | input {size}px | lr {lr} =====')
    for epoch in range(1, EPOCHS + 1):
        sync(); t0 = time.perf_counter()
        tr_loss, tr_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer, scaler)
        sync(); ep_time = time.perf_counter() - t0
        train_time += ep_time
        scheduler.step()

        va_loss, va_acc, _, _ = run_epoch(model, val_loader, criterion)
        history.append(dict(epoch=epoch, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=va_loss, val_acc=va_acc, epoch_time_sec=ep_time))
        print(f'  ep {epoch:02d}/{EPOCHS} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | '
              f'val loss {va_loss:.4f} acc {va_acc:.4f} | {ep_time:.1f}s')

        if va_acc > best_acc:
            best_acc, best_epoch, bad_epochs = va_acc, epoch, 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1
            if PATIENCE and bad_epochs >= PATIENCE:
                print(f'  early stopping at epoch {epoch}')
                break

    # ---- test with the best checkpoint ----
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    te_loss, te_acc, preds, tgts = run_epoch(model, test_loader, criterion)
    p_m, r_m, f_m, _ = precision_recall_fscore_support(tgts, preds, average='macro', zero_division=0)
    p_w, r_w, f_w, _ = precision_recall_fscore_support(tgts, preds, average='weighted', zero_division=0)

    # ---- inference time (batch=1, FP32) ----
    lat_gpu = measure_latency(copy.deepcopy(model), size, DEVICE) if DEVICE.type == 'cuda' else np.nan
    lat_cpu = measure_latency(copy.deepcopy(model).cpu(), size, torch.device('cpu'), 5, 30) \
              if MEASURE_CPU_LATENCY else np.nan

    # ---- save extras ----
    safe = label.replace(' ', '_').replace('/', '-')
    pd.DataFrame(history).to_csv(f'{OUT_DIR}/history/{safe}.csv', index=False)
    with open(f'{OUT_DIR}/reports/{safe}_report.txt', 'w') as f:
        f.write(classification_report(tgts, preds, target_names=classes, zero_division=0))
        f.write('\nConfusion matrix:\n')
        f.write(np.array2string(confusion_matrix(tgts, preds)))

    row = {
        'model': label,
        'accuracy': te_acc,
        'precision_macro': p_m, 'recall_macro': r_m, 'f1_macro': f_m,
        'precision_weighted': p_w, 'recall_weighted': r_w, 'f1_weighted': f_w,
        'test_loss': te_loss,
        'best_val_acc': best_acc, 'best_epoch': best_epoch,
        'train_time_sec': train_time,
        'train_time_per_epoch_sec': train_time / len(history),
        'inference_ms_gpu_bs1': lat_gpu,
        'inference_ms_cpu_bs1': lat_cpu,
        'model_size_mb': os.path.getsize(ckpt_path) / 1e6,
        'params_total': n_params, 'params_millions': n_params / 1e6, 'params_trainable': n_trainable,
        'learning_rate': lr, 'epochs_planned': EPOCHS, 'epochs_run': len(history),
        'batch_size': BATCH_SIZE, 'loss_function': LOSS_NAME, 'optimizer': OPTIM_NAME,
        'input_size': size, 'pretrained': PRETRAINED, 'mixed_precision': USE_AMP, 'augmentation': 'RRC+HFlip+Rot+ColorJitter+Perspective+Blur+Erasing',
        'num_classes': len(classes),
    }
    print(f'  >> TEST acc {te_acc:.4f} | P {p_m:.4f} R {r_m:.4f} F1 {f_m:.4f} | '
          f'GPU {lat_gpu:.1f} ms | CPU {lat_cpu:.1f} ms | {row["model_size_mb"]:.1f} MB')
    del model, optimizer, scaler
    return row

## Run all 17 models
Results are written to `results.csv` in `OUT_DIR` after each model. Re-running this cell skips models that are already finished.

In [ ]:
RESULTS_CSV = f'{OUT_DIR}/results.csv'
rows = pd.read_csv(RESULTS_CSV).to_dict('records') if os.path.exists(RESULTS_CSV) else []
done = {r['model'] for r in rows}

for label, cfg in MODELS.items():
    if label in done:
        print(f'Skipping {label} (already in results.csv)')
        continue
    try:
        rows.append(train_model(label, cfg))
        pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    except Exception:
        print(f'!! {label} FAILED (it will be retried on the next run):')
        traceback.print_exc()
    finally:
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

print('\nDone. Results:', RESULTS_CSV)
if _BAD_FILES:
    print(f'\n[WARNING] {len(_BAD_FILES)} unreadable image file(s) were skipped '
          f'(replaced with a black placeholder) during training/eval:')
    for p in sorted(_BAD_FILES):
        print('   ', p)
    print('Consider deleting or re-exporting these files, then rerunning for clean results.')

## Results

In [ ]:
df = pd.read_csv(RESULTS_CSV)
show_cols = ['model', 'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
             'train_time_sec', 'inference_ms_gpu_bs1', 'inference_ms_cpu_bs1',
             'model_size_mb', 'params_millions', 'learning_rate', 'epochs_run',
             'batch_size', 'loss_function']
pd.set_option('display.width', 250); pd.set_option('display.max_columns', None)
df.sort_values('accuracy', ascending=False)[show_cols].reset_index(drop=True).round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(2, 2, figsize=(16, 11))
d = df.sort_values('accuracy')

ax[0, 0].barh(d['model'], d['accuracy']); ax[0, 0].set_title('Test accuracy')
ax[0, 1].barh(d['model'], d['train_time_sec'] / 60); ax[0, 1].set_title('Training time (min)')

ax[1, 0].scatter(df['inference_ms_gpu_bs1'], df['accuracy'])
for _, r in df.iterrows():
    ax[1, 0].annotate(r['model'], (r['inference_ms_gpu_bs1'], r['accuracy']), fontsize=8)
ax[1, 0].set_xlabel('GPU latency, batch=1 (ms)'); ax[1, 0].set_ylabel('Accuracy'); ax[1, 0].set_title('Accuracy vs. latency')

ax[1, 1].scatter(df['params_millions'], df['accuracy'])
for _, r in df.iterrows():
    ax[1, 1].annotate(r['model'], (r['params_millions'], r['accuracy']), fontsize=8)
ax[1, 1].set_xscale('log'); ax[1, 1].set_xlabel('Parameters (millions, log)'); ax[1, 1].set_ylabel('Accuracy')
ax[1, 1].set_title('Accuracy vs. size')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/summary_plots.png', dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import os

df = pd.read_csv(RESULTS_CSV)
print(f"{len(df)} model(s) in results.csv")

fig, ax = plt.subplots(1, 2, figsize=(16, 6))
plotted = 0
for label in df['model']:
    path = f'{OUT_DIR}/history/{label.replace(" ", "_").replace("/", "-")}.csv'
    if not os.path.exists(path):
        print(f"missing history: {path}")
        continue
    h = pd.read_csv(path)
    ax[0].plot(h['epoch'], h['val_loss'], marker='o', label=label)   # marker makes 1-epoch runs visible
    ax[1].plot(h['epoch'], h['val_acc'], marker='o', label=label)
    plotted += 1

print(f"plotted {plotted} model(s)")
ax[0].set_title('Validation loss'); ax[1].set_title('Validation accuracy')
for a in ax:
    a.set_xlabel('epoch')
    a.set_xticks(sorted({int(x) for line in a.get_lines() for x in line.get_xdata()}))
if plotted:
    ax[1].legend(fontsize=7, ncol=2)
plt.tight_layout(); plt.show()